In [ ]:
import os
import re
from io import StringIO
from pathlib import Path
from itertools import permutations
from itertools import product
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import math
import seaborn as sns
import warnings
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [ ]:
import squidpy as sq
from scipy.spatial import KDTree

def KD_sub_search(arr, filter_ref, filter_query, neighbours=5, summary='min'):
    '''
    arr = main array, shape of image
    filter_idx = index of arr to search
    '''
    
    # subset array
    query_arr = arr[filter_query]
    ref_arr = arr[filter_ref]
    
    # build KDTree from only subset, query regular array
    kdtree=KDTree(ref_arr)
    dist,points=kdtree.query(query_arr, k=neighbours,p=2)
    
    if summary=='min':
        # calculate mean dist. for this query
        dist = [np.min(d) for d in dist] # remove self and calc. min 
    elif summary=='mean':
        dist = [np.mean(d) for d in dist] # remove self and calc. mean 
    else:
        print('Defaulting to min.')
        dist = [np.min(d) for d in dist] # remove self and calc. min 
    
    return dist

def get_celltype_distance(slide, type1, type2, background=1, quantile=0.5, summary='mean', summary_plot=False):
# slide: select_slide() output
# type 1 and 2: cell types to compare

    spotA='{}_spots'.format(type1)
    spotB='{}_spots'.format(type2)

    type1_perc = slide.obs[type1].quantile(quantile)
    type2_perc = slide.obs[type2].quantile(quantile)

    slide.obs[spotA] = np.where((slide.obs[type1]>=type1_perc) & (slide.obs[type1]>=background), 1, 0)
    slide.obs[spotB] = np.where((slide.obs[type2]>=type2_perc) & (slide.obs[type2]>=background), 1, 0)

    # capture only both interesting cell types
    slide_focus = slide[(slide.obs[spotA]==1) | (slide.obs[spotB]==1)]

    # coord array
    a = slide_focus.obs[['x', 'y']].to_numpy()
    indexA = slide_focus.obs[spotA]==1
    indexB = slide_focus.obs[spotB]==1

    # KDtree sub search, summarised 
    dist = KD_sub_search(a, indexB, indexA, summary=summary, neighbours=6)
    
    if summary_plot==True:
        # keep only the celltype/spots of interest
        slide_focus = slide_focus[slide_focus.obs[spotA]==1]
        slide_focus.obs['distance_to_{}'.format(type2)] = dist
        summary_plot = sc.pl.spatial(slide_focus, cmap='magma',
                                    color=[spotA, spotB, 'distance_to_{}'.format(type2)],
                                    ncols=5, size=1.3, show=False)
    else:
        summary_plot=None

    return {'summary_plot':summary_plot, 'average_dist':np.mean(dist), 'dist':dist}

def get_distance_to_ivy_gap(slide, type1, type2, background=1, quantile=0.5, summary='mean', summary_plot=False):
# slide: select_slide() output
# type 1 and 2: cell types to compare

    ivy_features = ['Cellular tumor',
                     'Hyperplastic blood vessels',
                     'Infiltrating tumor (grey matter)',
                     'Infiltrating tumor (white matter)',
                     'Leading edge (grey matter)',
                     'Leading edge (white matter)',
                     'Microvascular proliferation',
                     'Necrosis',
                     'Perinecrotic zone',
                     'Pseudopalisading cells around necrosis']

    if type1 in ivy_features:
        ivy_type = 'type1'
    elif type2 in ivy_features:
        ivy_type = 'type2'

    spotA='{}_spots'.format(type1)
    spotB='{}_spots'.format(type2)

    if ivy_type == 'type1':
        slide.obs[spotA] = np.where((slide.obs[type1]>0.5), 1, 0)
        state_perc = slide.obs[type2].quantile(quantile)
        slide.obs[spotB] = np.where((slide.obs[type2]>=state_perc) & (slide.obs[type2]>=background), 1, 0)
    elif ivy_type == 'type2':
        state_perc = slide.obs[type1].quantile(quantile)
        slide.obs[spotA] = np.where((slide.obs[type1]>=state_perc) & (slide.obs[type1]>=background), 1, 0)
        slide.obs[spotB] = np.where((slide.obs[type2]>0.5), 1, 0)
    
    # capture only both interesting cell types
    slide_focus = slide[(slide.obs[spotA]==1) | (slide.obs[spotB]==1)]

    # coord array
    a = slide_focus.obs[['x', 'y']].to_numpy()
    indexA = slide_focus.obs[spotA]==1
    indexB = slide_focus.obs[spotB]==1

    # KDtree sub search, summarised 
    dist = KD_sub_search(a, indexB, indexA, summary=summary)
    
    if summary_plot==True:
        # keep only the celltype/spots of interest
        slide_focus = slide_focus[slide_focus.obs[spotA]==1]
        slide_focus.obs['distance_to_{}'.format(type2)] = dist
        summary_plot = sc.pl.spatial(slide_focus, cmap='magma',
                                    color=[spotA, spotB, 'distance_to_{}'.format(type2)],
                                    ncols=5, size=1.3, show=False)
    else:
        summary_plot=None

    return {'summary_plot':summary_plot, 'average_dist':np.mean(dist), 'dist':dist}


# defining useful function
def select_slide_sp(adata, s, s_col='sample'):
    r""" Select data for one slide from the spatial anndata object.

    :param adata: Anndata object with multiple spatial samples
    :param s: name of selected sample
    :param s_col: column in adata.obs listing sample name for each location
    """

    slide = adata[adata.obs[s_col].isin([s]), :]
    s_keys = list(slide.uns['spatial'].keys())
    s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]

    slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}

    return slide

def extract_feature(adata, feature):

    ad_subset = adata[:, adata.var['feature_types']==feature]

    obs_names = adata.obs.index.tolist()
    var_names = ad_subset.var_names
    array = ad_subset.X.toarray()

    df = pd.DataFrame(
        array,
        index=obs_names,
        columns=var_names
    )

    return df

/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, Future

In [ ]:
directory = '../../../data/GBM_LEAP_annotations/web_atlas/anndata/'

# load spatial data
adata_sp = {}
for filename in os.listdir(directory):
    if 'h5ad' in filename:
        adata_sp[re.sub('.h5ad', '', filename)] = sc.read_h5ad(directory+filename)



/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/software/conda/users/gd11/scverse_new/lib/python3.10/site-packages/annd

# Spatial niches

In [ ]:
# prep for group with niches with similar names
sd_metadata = '../../../data/GBM_LEAP_annotations/'
annotation_hierarchy = pd.read_csv(sd_metadata+'annotation_hierarchy.tsv', sep='\t')
annotation_hierarchy = annotation_hierarchy[['annotation_granular', 'annotation_coarse']]
annotation_hierarchy["annotation_granular"] = [
    f"{i} (cell state)" if "Hypoxic" in i else i
    for i in annotation_hierarchy["annotation_granular"]
]

In [ ]:
#feature_type = 'Cell state abundances'

# Extract relevant features
spatial_obs = {}
niche_list_per_section = []
ivy_features_per_section = []
for section in adata_sp:
    # extract feature type quant data to add it to spatial obs; retrive fullres pixel coords
    obs_cts = extract_feature(adata_sp[section], 'Cell state abundances')

    obs_cts_coarse = []
    for ct in set(annotation_hierarchy['annotation_coarse']):
        cts = annotation_hierarchy[annotation_hierarchy['annotation_coarse']==ct]['annotation_granular'].tolist()
        cts = [i for i in cts if i in obs_cts.columns]
        s = obs_cts[cts].sum(axis=1)
        s.name = ct
        obs_cts_coarse.append(s)
    obs_cts_coarse = pd.concat(obs_cts_coarse, axis=1)
    obs_cts_coarse = obs_cts_coarse.rename(columns={'Proliferative':'Proliferative (cell state)'})

    obs_niche = extract_feature(adata_sp[section], 'Spatial niche abundances')
    obs = obs_cts_coarse.reset_index()\
        .merge(obs_niche.reset_index(), on = 'index')\
        .merge(pd.DataFrame(adata_sp[section].obsm['spatial'], columns = ['x', 'y'], 
                         index = adata_sp[section].obs.index).reset_index(), on = 'index')

    if 'Histopath annotation overlap' in set(adata_sp[section].var['feature_types']):
        obs_ivy = extract_feature(adata_sp[section], 'Histopath annotation overlap')
        obs = obs.reset_index()\
                .merge(obs_ivy.reset_index(), on = 'index')

        ivy_features_per_section.append(obs_ivy.columns.tolist())
    niche_list_per_section.append(obs_niche.columns.tolist())

    # get visium scaling factor
    # adjust for fixed naming conventions
    spatial_section = list(adata_sp[section].uns['spatial'].keys())[0]
    spot_diameter_pxl = adata_sp[section].uns['spatial'][spatial_section]['scalefactors']['spot_diameter_fullres']
    sf = 55.0 / spot_diameter_pxl

    # convert coords to microns
    obs["x"] = obs["x"] * sf
    obs["y"] = obs["y"] * sf

    # merge new obs with spatial anndata 
    if obs.columns[0] not in adata_sp[section].obs.columns:
        if 'x' not in adata_sp[section].obs.columns:
            adata_sp[section].obs = adata_sp[section].obs.reset_index().merge(
                obs, on = 'index'
            ).set_index('index')
        else:
            print('Skipping spatial coords...')
            adata_sp[section].obs = adata_sp[section].obs.reset_index().merge(
                obs.drop(['x', 'y'], axis=1), on = 'index'
            ).set_index('index')

    spatial_obs[section] = obs

In [ ]:
# malignant + niches
malignant_cts = ['AC progenitor-like 1',
 'AC progenitor-like 2',
 'Proliferative AC-OPC-like',
 'AC progenitor-like 3',
 'AC-gliosis-like 1',
 'AC-gliosis-like 2',
 'AC-gliosis-like 3',
 'AC-gliosis-like 4',
 'AC progenitor-like 4',
 'Gliosis-like',
 'Proliferative nIPC-like',
 'Hypoxic 1 (cell state)',
 'Hypoxic 2  (cell state)',
 'Proliferative NPC-OPC-like',
 'OPC-NPC-like 1',
 'OPC-NPC-like 2',
 'OPC-NPC-like 3',
 'NPC-neuronal-like 1',
 'NPC-neuronal-like 2',
 'NPC-neuronal-like 3',
 'NPC-neuronal-like 4',
 'NPC-neuronal-like 5',
 'OPC-neuronal-like',
 'OPC-like 1',
 'OPC-like 2',
 'OPC-like 3',
 'OPC-like 4',
 'OPC-like 5',]

# coarse version:
malignant_cts = sorted(list(set(annotation_hierarchy[annotation_hierarchy['annotation_granular'].isin(malignant_cts)]['annotation_coarse'])))
malignant_cts = [
    'Proliferative (cell state)' if i == 'Proliferative' else i
    for i in malignant_cts
]
niches = list(set([i for l in niche_list_per_section for i in l]))
ivy_features = list(set([i for l in ivy_features_per_section for i in l]))

In [ ]:
selected_clusters = malignant_cts+niches

In [ ]:
distance_comparison=[]
for section in tqdm(spatial_obs):
    clusters = [i for i in selected_clusters if i in adata_sp[section].obs.columns]
    
    for clusterA, clusterB in permutations(clusters, 2):
        test = get_celltype_distance(
                    slide=adata_sp[section],
                    type1=clusterA,
                    type2=clusterB,
                    summary = 'mean',
                    quantile = 0.75,
                    background = 4)
        distance_comparison.append(pd.DataFrame({
                        'donor_id':re.sub('-.*$', '', section),
                        'sample':section,
                        'clusterA':clusterA,
                        'clusterB':clusterB,
                        'distance':test['dist']}))

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [09:32<00:00,  5.90s/it]


In [ ]:
# for ivyGAP:
#for section in spatial_obs:
#    clusters = [i for i in selected_clusters if i in adata_sp[section].obs.columns]
#
#    if 'Histopath annotation overlap' in set(adata_sp[section].var['feature_types']):
#        ivy_feature_list = [i for i in ivy_features if i in adata_sp[section].obs.columns]
#        pairs = list(product(clusters, ivy_feature_list)) + list(product(ivy_feature_list, clusters))
#        for clusterA, clusterB in pairs:
#            test = get_distance_to_ivy_gap(
#                        slide=adata_sp[section],
#                        type1=clusterA,
#                        type2=clusterB,
#                        summary = 'mean',
#                        quantile = 0.75,
#                        background = 4)
#            distance_comparison.append(pd.DataFrame({
#                            'donor_id':re.sub('-.*$', '', section),
#                            'sample':section,
#                            'clusterA':clusterA,
#                            'clusterB':clusterB,
#                            'distance':test['dist']}))

In [ ]:
distance_comparison = pd.concat(distance_comparison)
# missing annotation in slide introduces inf
distance_comparison.replace([np.inf, -np.inf], np.nan, inplace=True) 
distance_comparison.dropna(inplace=True)

In [ ]:
savedir = '../../../data/GBM_LEAP_annotations/cell2location/distances/'
distance_comparison.to_csv(savedir+'distances_niches_cell_states_p75.csv', index = None)